In [ ]:
import backtrader as bt
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---

### 1. Pull historical data using yfinance or investpy for PSE stocks, or your existing forex data

In [ ]:
# Download EUR/USD historical data
dt = yf.download('EUR=X', start='2020-01-01', end='2026-01-01', interval='1d')
dt.head()

---

### 2. Compute the signals: for SMA crossover, `df['SMA50'] = df['Close'].rolling(50).mean()`

In [ ]:
# Getting the simple moving average (SMA) closing prices of EUR/USD from past 50 and 200 prices
dt['SMA50'] = dt['Close'].rolling(50).mean()
dt['SMA200'] = dt['Close'].rolling(200).mean()

dt.tail()

---

### 3. Generate a position column: 1 when long, 0 otherwise

In [ ]:
# Long when the 50-day SMA crosses above the 200-day SMA, exit when it crosses back
position_condition = dt['SMA50'] > dt['SMA200']
dt['Position'] = np.where(position_condition, 1, 0)

dt.tail()

---

### 4. Compute strategy returns: `df['Strategy_Return'] = df['Position'].shift(1) * df['Daily_Return']`
- The `.shift(1)` is critical — it prevents look-ahead bias (using tomorrow's signal to trade today)
- Add transaction cost friction: subtract 0.1–0.3% per trade to make it realistic

In [ ]:
dt['Daily_Return'] = np.log(dt['Close'] / dt['Close'].shift(1)).fillna(0)
dt['Strategy_Return'] = dt['Position'].shift(1) * dt['Daily_Return']
dt['Strategy_Return'] = dt['Strategy_Return'].fillna(0)

# Transaction costs
cost_per_trade = 0.001
trade_occurs = dt['Position'].diff().abs()
dt['Strategy_Return'] = dt['Strategy_Return'] - (trade_occurs * cost_per_trade)

dt.tail()

---

### 5. Compute and plot: cumulative returns of strategy vs. buy-and-hold benchmark

In [ ]:
# Cumulative sum of log returns
cumulative_log_returns = dt['Strategy_Return'].cumsum()

# Converting log returns to percentage
dt['Cumulative_Return'] = np.exp(cumulative_log_returns)

# 1. Compute Benchmark Cumulative Return (holding the asset 100% of the time)
# We take the cumulative sum of the raw daily returns and exponentiate it
dt['Benchmark_Cum_Return'] = np.exp(dt['Daily_Return'].cumsum())

# 2. Compute Strategy Cumulative Return 
dt['Strategy_Cum_Return'] = np.exp(dt['Strategy_Return'].cumsum())

# 3. Plotting the Comparison
plt.figure(figsize=(12, 6))

# Plot both lines
plt.plot(dt.index, dt['Benchmark_Cum_Return'], label='Buy-and-Hold Benchmark', color='gray', alpha=0.7, lw=1.5)
plt.plot(dt.index, dt['Strategy_Cum_Return'], label='50/200 SMA Strategy', color='blue', lw=2)

# Format the chart beautifully
plt.title('Strategy Performance vs. Buy-and-Hold Benchmark', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Growth of $1 (Multiplier)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)

# Ensure layout fits cleanly
plt.tight_layout()
plt.show()

---

### 6. Report: Total Return, Sharpe Ratio, Max Drawdown, Win Rate, number of trades

In [ ]:
# Total return
total_return = np.exp(dt['Strategy_Return'].sum()) - 1


# Sharpe Ratio
mean_return = dt['Strategy_Return'].mean()
volatility = dt['Strategy_Return'].std()

annualized_return = mean_return * 252
annualized_vol = volatility * np.sqrt(252)

rf = 0.01933 # Euro Short-Term Rate - https://www.ecb.europa.eu/stats/financial_markets_and_interest_rates/euro_short-term_rate/html/index.en.html
sharpe = (annualized_return - rf) / annualized_vol


# MDD
equity_curve = np.exp(dt['Strategy_Return'].cumsum())

# Compute the running peak
peaks = equity_curve.cummax()

# Compute the drawdown percentage
drawdowns = (equity_curve - peaks) / peaks

# Extract the worst (minimum) drawdown value
max_drawdown = drawdowns.min() * 100


# Win Rate
# .diff() shows where Position shifts from 0 to 1, or 1 to 0
dt['Trade_Signal'] = dt['Position'].diff()

# Grouping rows that belong to the same active trade window
dt['Trade_ID'] = (dt['Trade_Signal'] != 0).cumsum()

# Filter for rows where you actually hold a position (Position == 1)
active_trades = dt[dt['Position'] == 1]

# We sum the log returns for each trade window, then exponentiate
trade_performance = active_trades.groupby('Trade_ID')['Daily_Return'].sum().apply(np.exp) - 1

num_trades = len(trade_performance)
winning_trades = (trade_performance > 0).sum()

# Handle edge case where no trades were taken
win_rate = (winning_trades / num_trades) * 100 if num_trades > 0 else 0.0


print(f'Total Return: {total_return * 100:.3f}%')
print(f"Sharpe Ratio {sharpe:.3f}:")
print(f'Max Drawdown: {max_drawdown:.3f}%')
print(f"Win Rate: {win_rate:.3f}%")
print(f"Number of Trades: {num_trades}")

---

### 7. Interpretations

#### a. Total Return: -7.357%

In [ ]:
print('Initial price (2020-01-01):')
dt.iloc[1, 0]

In [ ]:
print('Final price (2025-12-31):')
dt.iloc[-1, 0]

The total return of EUR/USD itself, the benchmark (buying from day one and holding until the end), is `-4.477%`, and the strategy's total return is `-7.357%`. The strategy underperforms by `2.88%`, meaning that the strategy both failed to protect against the decline in EUR/USD and incurred additional losses. This indicates that for this asset and time period (6 years), **it would've been more beneficial to do nothing**.

#### b. Sharpe Ratio: -0.596

A negative Sharpe Ratio explains that the strategy took on volatility and **incurred losses greater than just investing in a risk-free savings account earning the Euro Short-Term Rate of `1.933%` per year** (European Central Bank, 2021). The strategy failed to compensate for the risk it assumed. **An investor would be better off in a risk-free instrument than in this strategy**.

#### c. Maximum Drawdown: -23.744%

From EUR/USD's peak value in the given time period, the strategy's equity fell by `23.744%` before recovering. To illustrate, on a `$1,000` starting portfolio, the balance would drop `$762.56`, and bouncing back to break even requires a `~31.1%` gain, **not just `23.744%`**. Most institutions set a maximum drawdown limit of `10%-20%`, so **the strategy would be shutdown in a professional setting** (TradeZella, 2026).

#### d. Win Rate: 33.333%

From 6 total trades, **the strategy became profitable only twice** (2 wins, 4 losses). However, this does not automatically disqualify the strategy. In a professional setting, 6 trades is **statistically meaningless**; a minimum of 30 trades is required before making any reliable conclusions about whether the 33.333% win rate can be attributed as a property of the strategy or just bad luck. 

### 8. Conclusion

The 50/200 SMA is a **trend-following strategy**, meaning that it waits for the signal of a sustained uptrend before entering the market, then it exits when the trend reverses. The EUR/USD from 2020 to 2026 was choppy, range-bound, and had no clear and sustained uptrend. Due to this, the strategy went long when there were signs of a trend, but the price quickly reversed shortly after like a **whipsaw**. This whipsaw pattern is what dominates the overall environment of the market at the given time period.

In conclusion, this strategy was applied to the wrong type of market, and testing it on a trending asset may produce very different results.

---

## References

European Central Bank. (2021, April 7). Euro short-term rate (€STR). European Central Bank. https://www.ecb.europa.eu/stats/financial_markets_and_interest_rates/euro_short-term_rate/html/index.en.html

TradeZella. (2026, April 14). Drawdown Management: How to Survive and Recover from Trading Drawdowns. Tradezella.com; TradeZella. https://www.tradezella.com/blog/drawdown-management

Yahoo Finance. (2025). Yahoo Finance - Business Finance, Stock Market, Quotes, News. Yahoo Finance. https://finance.yahoo.com/